In [ ]:
data_all_folder = '/home/xukang/2026_work/hic_50kb_processed'

In [ ]:
import os
import numpy as np
from typing import Any, Dict


def _find_matrix_64x64(d: Dict[str, Any]) -> np.ndarray:
    """从 dict 中提取出 (64,64) 的距离矩阵。"""
    if 'matrix' in d:
        m = d['matrix']
        if isinstance(m, np.ndarray) and m.shape == (64, 64):
            return m
    candidates = []
    for k, v in d.items():
        if isinstance(v, np.ndarray) and v.shape == (64, 64):
            candidates.append((k, v))
    if not candidates:
        raise KeyError("dict 中未找到 shape==(64,64) 的矩阵字段")
    if len(candidates) > 1:
        raise ValueError(f"dict 中找到多个 (64,64) 矩阵字段: {[k for k,_ in candidates]}")
    return candidates[0][1]


def convert_chr_file(src_path: str, dst_path: str, *, backup: bool = True) -> None:
    """把单个 chr*.npy（object 数组，元素为 dict）转为标准 (n,64,64) 数组并保存。"""
    arr = np.load(src_path, allow_pickle=True)

    if isinstance(arr, np.ndarray) and arr.dtype != object:
        if arr.ndim == 3 and arr.shape[1:] == (64, 64):
            out = arr.astype(np.float32, copy=False)
            np.save(dst_path, out)
            return
        if arr.ndim == 2 and arr.shape == (64, 64):
            out = arr.astype(np.float32, copy=False)[None, ...]
            np.save(dst_path, out)
            return

    if not (isinstance(arr, np.ndarray) and arr.dtype == object):
        raise TypeError(f"不支持的数组类型/格式: path={src_path}, type={type(arr)}, dtype={getattr(arr,'dtype',None)}")

    if arr.size == 0:
        raise ValueError(f"空数组: {src_path}")

    matrices = []
    for i in range(arr.shape[0]):
        item = arr[i]
        if not isinstance(item, dict):
            raise TypeError(f"第 {i} 个元素不是 dict: {src_path}, type={type(item)}")
        m = _find_matrix_64x64(item)
        matrices.append(m)

    out = np.stack(matrices, axis=0).astype(np.float32, copy=False)  # (n,64,64)

    if backup and os.path.abspath(dst_path) == os.path.abspath(src_path):
        bak_path = src_path + '.bak'
        if not os.path.exists(bak_path):
            os.rename(src_path, bak_path)

    np.save(dst_path, out)


def batch_convert(data_all_folder: str, *, overwrite: bool = True, backup: bool = True) -> None:
    """遍历 data_all_folder/*/chr*.npy 并批量转换为标准 (n,64,64)。"""
    if not os.path.isdir(data_all_folder):
        raise NotADirectoryError(f"data_all_folder 不存在: {data_all_folder}")

    subdirs = [os.path.join(data_all_folder, d) for d in os.listdir(data_all_folder)]
    subdirs = [p for p in subdirs if os.path.isdir(p)]

    total_files = 0
    converted = 0

    for sub in sorted(subdirs):
        for fn in sorted(os.listdir(sub)):
            if not (fn.startswith('chr') and fn.endswith('.npy')):
                continue
            src_path = os.path.join(sub, fn)

            # 默认：目标覆盖源；也可以改成写到新名字
            if overwrite:
                dst_path = src_path
            else:
                dst_path = os.path.join(sub, fn.replace('.npy', '.standard.npy'))

            total_files += 1
            try:
                # 跳过已是标准三维数组且正确的文件（可节省时间）
                test_arr = np.load(src_path, allow_pickle=True)
                if isinstance(test_arr, np.ndarray) and test_arr.dtype != object:
                    if test_arr.ndim == 3 and test_arr.shape[1:] == (64, 64):
                        continue

                convert_chr_file(src_path, dst_path, backup=backup)
                converted += 1
                print(f"[OK] {src_path} -> {dst_path}")
            except Exception as e:
                print(f"[FAIL] {src_path}: {e}")
                raise

    print(f"完成：总文件 {total_files}，实际转换 {converted}")





In [ ]:
output_root = '/home/xukang/2026_work/hic_50kb_processed2npy'


def batch_convert_to_output(data_all_folder: str, output_root: str, *, overwrite: bool = True) -> None:
    if not os.path.isdir(data_all_folder):
        raise NotADirectoryError(f"data_all_folder 不存在: {data_all_folder}")

    os.makedirs(output_root, exist_ok=True)

    subdirs = [
        os.path.join(data_all_folder, d)
        for d in os.listdir(data_all_folder)
        if os.path.isdir(os.path.join(data_all_folder, d))
    ]

    total_files = 0
    converted = 0
    skipped = 0

    for sub in sorted(subdirs):
        rel_sub = os.path.relpath(sub, data_all_folder)  # 保持原子目录名
        out_sub = os.path.join(output_root, rel_sub)
        os.makedirs(out_sub, exist_ok=True)

        for fn in sorted(os.listdir(sub)):
            if not (fn.startswith('chr') and fn.endswith('.npy')):
                continue

            src_path = os.path.join(sub, fn)
            dst_path = os.path.join(out_sub, fn)  # 文件名不变

            total_files += 1
            if (not overwrite) and os.path.exists(dst_path):
                skipped += 1
                continue

            # 如果 dst 已存在且已经是标准 (n,64,64)，则跳过（避免重复耗时）
            if os.path.exists(dst_path) and overwrite:
                try:
                    test_arr = np.load(dst_path, allow_pickle=False)
                    if isinstance(test_arr, np.ndarray) and test_arr.ndim == 3 and test_arr.shape[1:] == (64, 64):
                        skipped += 1
                        continue
                except Exception:
                    pass

            convert_chr_file(src_path, dst_path, backup=False)
            converted += 1
            print(f"[OK] {src_path} -> {dst_path}")

    print(f"完成：总文件 {total_files}，实际转换 {converted}，跳过 {skipped}")


batch_convert_to_output(data_all_folder, output_root, overwrite=True)


In [ ]:
import os
import numpy as np
from numpy.lib.format import open_memmap

input_root = '/home/xukang/2026_work/hic_50kb_processed2npy'
output_dir = '/home/xukang/2026_work/hic_50kb_all'
output_file = os.path.join(output_dir, 'all_chr_noY_64x64.npy')

overwrite = True  # 如果 output_file 已存在

os.makedirs(output_dir, exist_ok=True)
if overwrite and os.path.exists(output_file):
    os.remove(output_file)
if os.path.exists(output_file) and not overwrite:
    raise FileExistsError(f"输出文件已存在: {output_file}")

# 收集所有 chr*.npy（排除 chrY.npy）
paths = []
for sub in sorted(os.listdir(input_root)):
    sub_path = os.path.join(input_root, sub)
    if not os.path.isdir(sub_path):
        continue
    for fn in sorted(os.listdir(sub_path)):
        if not (fn.startswith('chr') and fn.endswith('.npy')):
            continue
        if fn == 'chrY.npy':
            continue
        paths.append(os.path.join(sub_path, fn))

if not paths:
    raise RuntimeError(f"未在 {input_root} 找到任何 chr*.npy（排除 chrY.npy 后为空）")

# 第一遍：统计总样本数，避免一次性把所有数据拼到内存
TOTAL = 0
for p in paths:
    arr = np.load(p, mmap_mode='r')
    if isinstance(arr, np.ndarray) and arr.ndim == 3:
        if arr.shape[1:] != (64, 64):
            raise ValueError(f"维度异常: {p}, shape={arr.shape}")
        n = arr.shape[0]
    elif isinstance(arr, np.ndarray) and arr.ndim == 2:
        if arr.shape != (64, 64):
            raise ValueError(f"维度异常: {p}, shape={arr.shape}")
        n = 1
    else:
        raise ValueError(f"非标准数组: {p}, shape={getattr(arr,'shape',None)}, ndim={getattr(arr,'ndim',None)}")
    TOTAL += int(n)

print(f"待合并文件数: {len(paths)}，总样本数 TOTAL={TOTAL}")

# 第二遍：memmap 方式写入，节省内存
mm = open_memmap(output_file, mode='w+', dtype=np.float32, shape=(TOTAL, 64, 64))

cursor = 0
for i, p in enumerate(paths, 1):
    arr = np.load(p, mmap_mode='r')
    if arr.ndim == 2:
        arr = arr[None, ...]

    n = arr.shape[0]
    mm[cursor:cursor+n] = arr.astype(np.float32, copy=False)
    cursor += n

    if i % 50 == 0 or i == len(paths):
        print(f"写入进度: {i}/{len(paths)}，cursor={cursor}")

mm.flush()

# 简单校验
check = np.load(output_file, mmap_mode='r')
print('已保存:', output_file)
print('校验 shape/dtype:', check.shape, check.dtype)


In [ ]:
# ========== 过滤：去掉 0 值占比 > 90% 的矩阵 ==========
import os
import numpy as np
from numpy.lib.format import open_memmap

input_file = '/home/xukang/2026_work/hic_50kb_all/all_chr_noY_64x64.npy'
output_dir = '/home/xukang/2026_work/hic_50kb_all'

zero_ratio_threshold = 0.90  # 0 值占比 > 90% 的剔除
chunk_size = 512  # 每次处理多少个矩阵，控制内存

os.makedirs(output_dir, exist_ok=True)
base = os.path.splitext(os.path.basename(input_file))[0]
output_file = os.path.join(output_dir, f'{base}_zero_ratio_le_{int(zero_ratio_threshold*100)}.npy')
kept_idx_file = os.path.join(output_dir, f'{base}_kept_indices_zero_ratio_le_{int(zero_ratio_threshold*100)}.npy')
zero_ratio_file = os.path.join(output_dir, f'{base}_zero_ratios.npy')

if os.path.exists(output_file):
    print(f"输出已存在: {output_file}，将跳过过滤。")
else:
    # 载入为 memmap，避免一次性读入内存
    X = np.load(input_file, mmap_mode='r')
    if not (isinstance(X, np.ndarray) and X.ndim == 3 and X.shape[1:] == (64, 64)):
        raise ValueError(f"输入形状不符合预期: {input_file}, shape={getattr(X,'shape',None)}")

    N = X.shape[0]
    print('Input N =', N)

    zero_ratios = np.empty((N,), dtype=np.float32)
    keep_mask = np.empty((N,), dtype=np.bool_)

    # -------- 第一遍：计算每个矩阵的 0 值占比 --------
    for start in range(0, N, chunk_size):
        end = min(N, start + chunk_size)
        block = X[start:end]  # (B,64,64)

        # 0 值占比（按元素精确等于 0）
        zratio = (block == 0).mean(axis=(1, 2))
        zero_ratios[start:end] = zratio.astype(np.float32, copy=False)
        keep_mask[start:end] = zratio <= zero_ratio_threshold

        if (start // chunk_size) % 20 == 0:
            print(f"进度: {start}/{N}")

    kept_indices = np.nonzero(keep_mask)[0].astype(np.int32)
    kept_count = int(kept_indices.shape[0])
    removed_count = N - kept_count
    print('kept:', kept_count, 'removed:', removed_count)

    # 保存辅助信息
    np.save(kept_idx_file, kept_indices)
    np.save(zero_ratio_file, zero_ratios)

    # -------- 第二遍：把保留的矩阵写入新文件 --------
    mm = open_memmap(output_file, mode='w+', dtype=np.float32, shape=(kept_count, 64, 64))

    write_cursor = 0
    for start in range(0, N, chunk_size):
        end = min(N, start + chunk_size)

        idx_local = np.nonzero(keep_mask[start:end])[0]
        if idx_local.size == 0:
            continue

        block = X[start:end]
        # block[idx_local] 形状：(B_keep,64,64)
        selected = block[idx_local].astype(np.float32, copy=False)
        bsz = selected.shape[0]

        mm[write_cursor:write_cursor+bsz] = selected
        write_cursor += bsz

        print(f"写入进度: cursor={write_cursor}/{kept_count}")

    mm.flush()

    # -------- 简单校验 --------
    check = np.load(output_file, mmap_mode='r')
    print('输出完成:', output_file)
    print('check shape/dtype:', check.shape, check.dtype)


In [ ]:
# ========== ICE归一化 -> log -> minmax(0-1) -> 对角线置1 ==========
import os
import numpy as np
from numpy.lib.format import open_memmap

filtered_input = '/home/xukang/2026_work/hic_50kb_all/all_chr_noY_64x64_zero_ratio_le_90.npy'

# 输出文件（同目录下）
out_dir = os.path.dirname(filtered_input)
base = os.path.splitext(os.path.basename(filtered_input))[0]
output_file = os.path.join(out_dir, f'{base}_ice_log_minmax_diag1.npy')

# ICE参数：迭代次数越多越接近，但越慢
max_iter = 5
# 允许提前停止（相对偏差阈值，通常可设 1e-3~1e-4）
tol = 1e-3
# 处理数值稳定
eps = 1e-8

chunk_size = 512  # 建议 256~1024 之间

if os.path.exists(output_file):
    print('输出已存在，跳过：', output_file)
else:
    X = np.load(filtered_input, mmap_mode='r')
    if not (isinstance(X, np.ndarray) and X.ndim == 3 and X.shape[1:] == (64, 64)):
        raise ValueError(f'输入形状不符合预期: {filtered_input}, shape={getattr(X,"shape",None)}')

    N = X.shape[0]
    print('Input N =', N)

    mm = open_memmap(output_file, mode='w+', dtype=np.float32, shape=(N, 64, 64))

    idx = np.arange(64)

    def ice_normalize_inplace(batch: np.ndarray) -> np.ndarray:
        """对称矩阵批处理ICE：通过行和列缩放迭代平衡。batch shape=(B,64,64)。"""
        # 假设输入为非负；若有负数会在 log 之前通过 clip 修正。
        for it in range(max_iter):
            row_sums = batch.sum(axis=2)  # (B,64)
            scale = np.zeros_like(row_sums, dtype=batch.dtype)
            mask = row_sums > eps
            scale[mask] = 1.0 / row_sums[mask]

            # 行缩放
            batch *= scale[:, :, None]
            # 列缩放（保持对称性）
            batch *= scale[:, None, :]

            # 收敛判断：行和接近其均值
            row_sums2 = batch.sum(axis=2)
            mean = row_sums2.mean(axis=1, keepdims=True)
            rel_dev = np.mean(np.abs(row_sums2 - mean) / (mean + eps), axis=1)  # (B,)
            if float(np.max(rel_dev)) < tol:
                break
        return batch

    out_cursor = 0
    for start in range(0, N, chunk_size):
        end = min(N, start + chunk_size)
        b = end - start

        # 取chunk到内存（ICE与log需要在内存里做）
        batch = np.asarray(X[start:end], dtype=np.float32).copy()

        # 1) ICE归一化
        batch = ice_normalize_inplace(batch)

        # 2) 取log（避免 log(0)）
        batch = np.log(np.clip(batch, eps, None))

        # 3) 每个矩阵 min-max 到 0-1
        x_min = batch.min(axis=(1, 2), keepdims=True)
        x_max = batch.max(axis=(1, 2), keepdims=True)
        denom = x_max - x_min
        denom = np.where(denom > eps, denom, 1.0)  # 避免除0
        batch = (batch - x_min) / denom

        # 4) 对角线置1
        batch[:, idx, idx] = 1.0

        mm[start:end] = batch
        print(f'进度: {end}/{N}')

    mm.flush()
    print('输出完成：', output_file)
